In [1]:
# single branch GRU training script
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import MinMaxScaler
import argparse
import sys
import os
import matplotlib.pyplot as plt

src_path = os.path.abspath(os.path.join(os.getcwd(), 'many_to_many', 'src'))
if src_path not in sys.path:
    sys.path.append(src_path)
    
from utils import create_sliding_windows_m2m, SequentialDeepONetDataset, train_val_test_split
from s_deeponet import SequentialDeepONet

torch.manual_seed(0)
np.random.seed(0)

In [2]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

   Static hostname: gpua061.delta.ncsa.illinois.edu
         Icon name: computer-server
           Chassis: server
        Machine ID: 4216c320c33b02efc2510a15f446eb32
           Boot ID: 1336515662fd49beb5e803cc76cd9caf
  Operating System: ]8;;https://www.redhat.com/Red Hat Enterprise Linux 8.8 (Ootpa)]8;;
       CPE OS Name: cpe:/o:redhat:enterprise_linux:8::baseos
            Kernel: Linux 4.18.0-477.95.1.el8_8.x86_64
      Architecture: x86-64
Using device: cuda


In [3]:
data_path = '/path/Cosmic10000/data'

target_file = 'dose/combined.npy'
branch_file = 'neutron/neutron_data_2001_2023.npy'
trunc_file = 'coord/grid_array.npy'
target = np.load(os.path.join(data_path,target_file))
input_data = np.load(os.path.join(data_path,branch_file))
trunk = np.load(os.path.join(data_path,trunc_file))

In [5]:
# Normalize trunk input
trunk[:, 0] = (trunk[:, 0] - np.min(trunk[:, 0])) / (np.max(trunk[:, 0]) - np.min(trunk[:, 0]))
trunk[:, 1] = (trunk[:, 1] - np.min(trunk[:, 1])) / (np.max(trunk[:, 1]) - np.min(trunk[:, 1]))

# Assuming input_data and target are defined elsewhere in the notebook
train_input, train_target, val_input, val_target, test_input, test_target = train_val_test_split(input_data, target)

Train input shape: (4017, 12)
Validation input shape: (4018, 12)
Test input shape: (365, 12)


In [6]:
# input data normalization (min-max scaling)
scaler = MinMaxScaler()

train_input = scaler.fit_transform(train_input)
# val_input = scaler.transform(val_input)
test_input = scaler.transform(test_input)

In [7]:
# target data normalization (min-max scaling)
scaler_target = MinMaxScaler()

train_target = scaler_target.fit_transform(train_target)[..., np.newaxis]
# val_target = scaler_target.transform(val_target)[..., np.newaxis]
test_target = scaler_target.transform(test_target)[..., np.newaxis]

In [8]:
window_size = 14
pred_window = 14

## Generate sequences for the training set
# train_input_seq, train_target_seq = create_sliding_windows_m2m(train_input, train_target, window_size, pred_window = pred_window)

## generate sequences for the validation set
# val_input_seq, val_target_seq = create_sliding_windows_m2m(val_input, val_target, window_size, pred_window = pred_window)

# Generate sequences for the testing set
test_input_seq, test_target_seq = create_sliding_windows_m2m(test_input, test_target, window_size, pred_window = pred_window)

# print the shapes of the generated sequences
print("Check the shapes of the generated sequences\n-----------------------------------------")
# print("Train input shape:", train_input_seq.shape)
# print("Train target shape:", train_target_seq.shape)
# print("Validation input shape:", val_input_seq.shape)
# print("Validation target shape:", val_target_seq.shape)
print("Test input shape:", test_input_seq.shape)
print("Test target shape:", test_target_seq.shape)
print("-----------------------------------------")

Check the shapes of the generated sequences
-----------------------------------------
Test input shape: torch.Size([338, 14, 12])
Test target shape: torch.Size([338, 14, 65341, 1])
-----------------------------------------


In [9]:
# Create DataLoaders for training and validation sets
print("Create DataLoaders for training and validation sets\n-----------------------------------------")
batch_size = 16
print("Batch size:", batch_size)

# train_dataset = SequentialDeepONetDataset(train_input_seq, trunk, train_target_seq)
# train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True, drop_last=True)

# val_dataset = SequentialDeepONetDataset(val_input_seq, trunk, val_target_seq)
# val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

test_dataset = SequentialDeepONetDataset(test_input_seq, trunk, test_target_seq)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

Create DataLoaders for training and validation sets
-----------------------------------------
Batch size: 16


In [12]:
p, num_outs = test_target_seq.shape[-2], test_target_seq.shape[-1]

# %%
def init_model(model_type):
    dim = 128
    model = SequentialDeepONet(
        branch_type=model_type,
        branch_input_size=12,
        branch_hidden_size=128,
        branch_num_layers=4,
        branch_output_size=dim,
        trunk_architecture=[2, 128, 128, dim],
        num_outputs=num_outs,
        activation_fn=nn.ReLU,
        pred_window = pred_window
    )
    return model

In [13]:
def test_and_save_model(model_type, window_size, pred_window, device, test_loader, scaler_target, model_path, save_path):
    model = init_model(model_type)
    model.load_state_dict(torch.load(f'{model_path}/{model_type}_best_model_10km_lrschd_{window_size}_{pred_window}.pth'))
    model = model.to(device)
    model.eval()

    pred = []
    gt = []

    with torch.no_grad():
        for batch in test_loader:
            x_branch, x_trunk, y = batch
            x_branch = x_branch.to(device)
            x_trunk = x_trunk.to(device)

            output = model(x_branch, x_trunk)
            output_cpu = output.detach().cpu()
            pred.append(output_cpu)
            gt.append(y.cpu())

            del x_branch, x_trunk, output
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    pred = torch.cat(pred, dim=0).numpy()
    gt = torch.cat(gt, dim=0).numpy()

    # Reshape and inverse transform
    batch_size, channels, spatial_points, time_steps = pred.shape
    all_preds = pred.reshape(batch_size * channels, spatial_points, time_steps)
    all_targets = gt.reshape(batch_size * channels, spatial_points, time_steps)

    all_preds_2d = all_preds.reshape(all_preds.shape[0], -1)
    all_targets_2d = all_targets.reshape(all_targets.shape[0], -1)

    all_preds_inverse = scaler_target.inverse_transform(all_preds_2d)
    all_targets_inverse = scaler_target.inverse_transform(all_targets_2d)

    all_preds_final = all_preds_inverse.reshape(batch_size, channels, spatial_points, time_steps)
    all_targets_final = all_targets_inverse.reshape(batch_size, channels, spatial_points, time_steps)

    # Save results
    np.save(f'{save_path}/{model_type}_predictions_{window_size}_{pred_window}.npy', all_preds_final)
    np.save(f'{save_path}/{model_type}_targets_{window_size}_{pred_window}.npy', all_targets_final)

    return all_preds_final, all_targets_final

# Usage
save_path = '/inference_results'
model_path = '/saved_models'
model_types = ['gru', 'lstm', 'fcn', 'transformer']
window_size = 14  # Replace with your actual window size
pred_window = 14   # Replace with your actual prediction window

for model_type in model_types:
    test_and_save_model(model_type, window_size, pred_window, device, test_loader, scaler_target, model_path, save_path)